## Imports

In [12]:
import wandb
import logging
from tqdm import tqdm
from wandb.sdk.wandb_run import Run
import numpy as np
import pandas as pd
import plotly.express as px
import numpy as np
import plotly.graph_objs as go
import seaborn as sns
import matplotlib.pyplot as plt
from nn_core.common import PROJECT_ROOT
import json

## Configuration

In [13]:
from mass.utils.plots import Palette

plt.rcParams.update(
    {
        "text.usetex": True,
        "font.family": "serif",
        "axes.titlesize": 24,  # Larger axes/title fonts
        "axes.labelsize": 24,
        "xtick.labelsize": 24,
        "ytick.labelsize": 20,
        "legend.fontsize": 24,
    }
)
sns.set_context("talk")

cmap_name = "coolwarm_r"

palette = Palette(
    f"{PROJECT_ROOT}/misc/palette.json", map_path=f"{PROJECT_ROOT}/misc/palette_map.json"
)
palette

{'blue': '#335c67',
 'white': '#fff3b0',
 'yellow': '#e09f3e',
 'red': '#9e2a2b',
 'dark red': '#540b0e',
 'green': '#81b29a'}

## Get runs

In [14]:
api = wandb.Api()
entity, project = "gladia", "mass"  # set to your entity and project

In [15]:
def get_runs(entity, project, positive_tags, negative_tags):
    filters_pos_tags = {"$and": [{"tags": {"$eq": pos_tag}} for pos_tag in positive_tags]}
    filters_neg_tags = {}

    print(filters_pos_tags)
    filters = {**filters_pos_tags, **filters_neg_tags}
    runs = api.runs(entity + "/" + project, filters=filters)

    print(f"There are {len(runs)} runs respecting these conditions.")
    return runs

In [16]:
tags = ["Consensus", "benchmark"]

In [17]:
runs = get_runs(entity, project, positive_tags=tags, negative_tags=[])

{'$and': [{'tags': {'$eq': 'Consensus'}}, {'tags': {'$eq': 'benchmark'}}]}


There are 9 runs respecting these conditions.


In [18]:
ref_run = runs[0]

In [19]:
print(set(ref_run.history().columns))

{'acc/test/avg', 'loss/test/DTD', 'trainer/global_step', 'acc/test/EMNIST', 'normalized_acc/test/OxfordIIITPet', 'acc/test/PCAM', 'normalized_acc/test/Food101', 'loss/test/GTSRB', 'loss/test/SUN397', 'normalized_acc/test/GTSRB', 'acc/test/EuroSAT', 'normalized_acc/test/DTD', 'acc/test/SUN397', 'loss/test/FER2013', 'acc/test/RenderedSST2', 'normalized_acc/test/PCAM', 'loss/test/EuroSAT', 'loss/test/RenderedSST2', 'normalized_acc/test/MNIST', 'acc/test/GTSRB', 'loss/test/KMNIST', 'normalized_acc/test/RenderedSST2', 'normalized_acc/test/avg', 'acc/test/MNIST', 'loss/test/STL10', 'loss/test/CIFAR100', '_step', 'acc/test/CIFAR10', 'acc/test/SVHN', 'loss/test/MNIST', 'acc/test/Flowers102', 'normalized_acc/test/CIFAR100', '_timestamp', 'normalized_acc/test/Cars', 'loss/test/EMNIST', 'normalized_acc/test/CIFAR10', 'acc/test/FER2013', 'normalized_acc/test/FER2013', 'loss/test/FashionMNIST', 'loss/test/CIFAR10', 'normalized_acc/test/STL10', 'loss/test/Food101', 'normalized_acc/test/SUN397', 'nor

In [20]:
print(ref_run.config["core/tags"])

['Consensus', 'benchmark', 'static_merge', 'n20', 'ViT-L-14']


#### Hparams

In [21]:
benchmarks = ["n8", "n14", "n20"]
models = ["ViT-B-32", "ViT-B-16", "ViT-L-14"]

In [22]:
avg_accs = {
    model: {benchmark: {"avg_acc": 0.0, "norm_acc": 0.0} for benchmark in benchmarks}
    for model in models
}

for run in runs:
    model = run.config["nn/encoder/model_name"]

    try:
        N = run.config["num_tasks"]
    except KeyError:
        N = run.config["ntasks"]

    benchmark = f"n{N}"

    avg_accs[model][benchmark]["avg_acc"] = run.summary["acc/test/avg"]
    avg_accs[model][benchmark]["norm_acc"] = run.summary["normalized_acc/test/avg"]

In [23]:
avg_accs

{'ViT-B-32': {'n8': {'avg_acc': 0.7258774638175964,
   'norm_acc': 0.8008057698607445},
  'n14': {'avg_acc': 0.7031656929424831, 'norm_acc': 0.7886840871402195},
  'n20': {'avg_acc': 0.685051691532135, 'norm_acc': 0.7678791016340256}},
 'ViT-B-16': {'n8': {'avg_acc': 0.7587374374270439,
   'norm_acc': 0.8165058344602585},
  'n14': {'avg_acc': 0.7488678310598645, 'norm_acc': 0.8169107522283282},
  'n20': {'avg_acc': 0.7219799026846886, 'norm_acc': 0.7840635314583778}},
 'ViT-L-14': {'n8': {'avg_acc': 0.8550726398825645,
   'norm_acc': 0.9050783663988112},
  'n14': {'avg_acc': 0.8203247700418744, 'norm_acc': 0.8761024304798671},
  'n20': {'avg_acc': 0.7885549522936344, 'norm_acc': 0.8379229836165905}}}

In [24]:
# print latex

row = "& "
for model in models:

    for benchmark in benchmarks:
        avg_acc = avg_accs[model][benchmark]["avg_acc"]
        norm_acc = avg_accs[model][benchmark]["norm_acc"]

        row += f"${avg_acc*100:.1f}_{{({norm_acc*100:.1f})}}$ & "


print(row[:-2] + "\\\\")

& $72.6_{(80.1)}$ & $70.3_{(78.9)}$ & $68.5_{(76.8)}$ & $75.9_{(81.7)}$ & $74.9_{(81.7)}$ & $72.2_{(78.4)}$ & $85.5_{(90.5)}$ & $82.0_{(87.6)}$ & $78.9_{(83.8)}$ \\
